In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
from typing_extensions import TypedDict
from typing import Any, Annotated
import operator

class State(TypedDict):
    state: Annotated[list, operator.add]

In [3]:
class ReturnNodeValue:
    def __init__(self, node_secret: str):
        print("Inside Constructor", node_secret)
        self.value = node_secret
        
    def __call__(self, state:State)-> Any:
        print(f"Adding {self.value} to state {state['state']}")
        return {"state":[self.value]}

In [4]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
builder.add_node("a", ReturnNodeValue("Node A"))
builder.add_node("b", ReturnNodeValue("Node B"))
builder.add_node("c", ReturnNodeValue("Node C"))
builder.add_node("d", ReturnNodeValue("Node D"))

builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", "c")
builder.add_edge("c", "d")
builder.add_edge("d", END)

graph = builder.compile()

Inside Constructor Node A
Inside Constructor Node B
Inside Constructor Node C
Inside Constructor Node D


In [5]:
graph.invoke({"state":[]})

Adding Node A to state []
Adding Node B to state ['Node A']
Adding Node C to state ['Node A', 'Node B']
Adding Node D to state ['Node A', 'Node B', 'Node C']


{'state': ['Node A', 'Node B', 'Node C', 'Node D']}

In [6]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
builder.add_node("a", ReturnNodeValue("Node A"))
builder.add_node("b", ReturnNodeValue("Node B"))
builder.add_node("c", ReturnNodeValue("Node C"))
builder.add_node("d", ReturnNodeValue("Node D"))

builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "d")
builder.add_edge("c", "d")
builder.add_edge("d", END)

graph = builder.compile()

Inside Constructor Node A
Inside Constructor Node B
Inside Constructor Node C
Inside Constructor Node D


In [7]:
graph.invoke({"state":[]})

Adding Node A to state []
Adding Node B to state ['Node A']
Adding Node C to state ['Node A']
Adding Node D to state ['Node A', 'Node B', 'Node C']


{'state': ['Node A', 'Node B', 'Node C', 'Node D']}

# Parallelization Example

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0, google_api_key=os.getenv("GEMINI_API_KEY"))

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class State(TypedDict):
    question: str
    context: Annotated[list, operator.add]
    answer: str

In [10]:
from langchain_community.tools import TavilySearchResults
from langchain_community.document_loaders import WikipediaLoader

def search_web(state):
    """ Retrieve docs from web search """
    tavily_search = TavilySearchResults(max_results=3)
    search_docs = tavily_search.invoke(state['question'])
    
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document href={doc["url"]}/>\n{doc["content"]}\n</Document>'
            for doc in search_docs
        ]
    )
    
    
    return {"context":[formatted_search_docs]}


def search_wikipedia(state):
    """ Retrieve docs from wikipedia """
    
    search_docs = WikipediaLoader(query=state['question'], load_max_docs=3).load()
    
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document href={doc.metadata["source"]}" page="{doc.metadata.get("page", "")}" />\n{doc.page_content}\n</Document>'
            for doc in search_docs
        ]
    )
    
    return {"context":[formatted_search_docs]}

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage

def generate_answer(state):
    """Node to answer a question"""
    
    context = state['context']
    question = state['question']
    question = state['question']
    
    answer_template = f"""Answer the question {question} using this context: {context}"""
    answer_instructions = answer_template.format(question=question, context=context)
    
    answer = llm.invoke([SystemMessage(content=answer_instructions)]+[HumanMessage(content="Answer the question based on the context provided.")])
    
    return {"answer":answer}

In [12]:
builder = StateGraph(State)

builder.add_node("search_web", search_web)
builder.add_node("search_wikipedia", search_wikipedia)
builder.add_node("generate_answer", generate_answer)

builder.add_edge(START, "search_web")
builder.add_edge(START, "search_wikipedia")

builder.add_edge("search_web", "generate_answer")
builder.add_edge("search_wikipedia", "generate_answer")
builder.add_edge("generate_answer", END)

graph = builder.compile()

In [13]:
result = graph.invoke({"question": "How were NVIDIA's earnings last quarter?"})
result['answer'].content

C:\Users\Aditya\AppData\Local\Temp\ipykernel_19972\1303254667.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=3)


"NVIDIA's earnings for the third quarter of fiscal year 2026 were:\n\n*   **Revenue:** $57.0 billion\n*   **GAAP Diluted Earnings Per Share:** $1.30\n*   **Non-GAAP Diluted Earnings Per Share:** $1.30\n*   **GAAP Gross Margin:** 73.4%\n*   **Non-GAAP Gross Margin:** 73.6%"